<a href="https://colab.research.google.com/github/TASRUNNER2010/Hackathon/blob/main/Hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/veritas AI project/ad_viz_plotval_data.csv')
df

,Date,Source,Site ID,POC,Daily Mean PM2.5 Concentration,Units,Daily AQI Value,Local Site Name,Daily Obs Count,Percent Complete,...,Method Code,Method Description,CBSA Code,CBSA Name,State FIPS Code,State,County FIPS Code,County,Site Latitude,Site Longitude
0,01/03/2023,AQS,340230011,1,12.6,ug/m3 LC,58,Rutgers University,1,100.0,...,145,R & P Model 2025 PM-2.5 Sequential Air Sampler...,35620,"New York-Newark-Jersey City, NY-NJ-PA",34,New Jersey,23,Middlesex,40.462182,-74.429439
1,01/06/2023,AQS,340230011,1,4.0,ug/m3 LC,22,Rutgers University,1,100.0,...,145,R & P Model 2025 PM-2.5 Sequential Air Sampler...,35620,"New York-Newark-Jersey City, NY-NJ-PA",34,New Jersey,23,Middlesex,40.462182,-74.429439
2,01/09/2023,AQS,340230011,1,7.6,ug/m3 LC,42,Rutgers University,1,100.0,...,145,R & P Model 2025 PM-2.5 Sequential Air Sampler...,35620,"New York-Newark-Jersey City, NY-NJ-PA",34,New Jersey,23,Middlesex,40.462182,-74.429439
3,01/12/2023,AQS,340230011,1,5.5,ug/m3 LC,31,Rutgers University,1,100.0,...,145,R & P Model 2025 PM-2.5 Sequential Air Sampler...,35620,"New York-Newark-Jersey City, NY-NJ-PA",34,New Jersey,23,Middlesex,40.462182,-74.429439
4,01/15/2023,AQS,340230011,1,2.2,ug/m3 LC,12,Rutgers University,1,100.0,...,145,R & P Model 2025 PM-2.5 Sequential Air Sampler...,35620,"New York-Newark-Jersey City, NY-NJ-PA",34,New Jersey,23,Middlesex,40.462182,-74.429439
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
452,12/26/2023,AQS,340230011,3,15.3,ug/m3 LC,63,Rutgers University,1,100.0,...,183,Thermo Scientific 5014i or FH62C14-DHS w/VSCC,35620,"New York-Newark-Jersey City, NY-NJ-PA",34,New Jersey,23,Middlesex,40.462182,-74.429439
453,12/27/2023,AQS,340230011,3,8.5,ug/m3 LC,47,Rutgers University,1,100.0,...,183,Thermo Scientific 5014i or FH62C14-DHS w/VSCC,35620,"New York-Newark-Jersey City, NY-NJ-PA",34,New Jersey,23,Middlesex,40.462182,-74.429439
454,12/28/2023,AQS,340230011,3,1.9,ug/m3 LC,11,Rutgers University,1,100.0,...,183,Thermo Scientific 5014i or FH62C14-DHS w/VSCC,35620,"New York-Newark-Jersey City, NY-NJ-PA",34,New Jersey,23,Middlesex,40.462182,-74.429439
455,12/29/2023,AQS,340230011,3,3.3,ug/m3 LC,18,Rutgers University,1,100.0,...,183,Thermo Scientific 5014i or FH62C14-DHS w/VSCC,35620,"New York-Newark-Jersey City, NY-NJ-PA",34,New Jersey,23,Middlesex,40.462182,-74.429439


In [14]:
# Save an independent copy of the original dataset before cleaning.
raw_df = df.copy()

# Remove extra spaces from the beginning and end of column names.
df.columns = df.columns.str.strip()

# Keep only the Rutgers University monitoring site (340230011)
# and instrument POC 3, which measures pollution mostly every day.
df = df.loc[
    (df["Site ID"].astype(str).str.strip() == "340230011")
    & (pd.to_numeric(df["POC"], errors="raise") == 3)
].copy()

# Keep the date, PM2.5 concentration, and AQI.
# Other columns describe the station or instrument and are not needed here.
df = df[[
    "Date",
    "Daily Mean PM2.5 Concentration",
    "Daily AQI Value"
]].copy()

# Rename columns to make them easier to use in Python.
# PM2.5 is measured in micrograms per cubic meter.
df = df.rename(columns={
    "Date": "date",
    "Daily Mean PM2.5 Concentration": "pm25",
    "Daily AQI Value": "aqi"
})

# Convert date strings such as "01/03/2023" into actual dates.
# The specified format means month/day/year.
df["date"] = pd.to_datetime(
    df["date"],
    format="%m/%d/%Y",
    errors="raise"
)

# Ensure pollution and AQI values are numeric.
# Stop with an error if an unexpected nonnumeric value appears.
for column in ["pm25", "aqi"]:
    df[column] = pd.to_numeric(df[column], errors="raise")

# Remove rows that repeat the exact same date, PM2.5, and AQI.
df = df.drop_duplicates()

# Stop if a date still has multiple different measurements.
# These would need investigation rather than arbitrary deletion.
if df["date"].duplicated().any():
    raise ValueError("Multiple different measurements remain for the same date.")

# Stop if filtering unexpectedly removed all measurements.
if df.empty:
    raise ValueError("No measurements found for the selected site and instrument.")

# Use dates as the row index and sort from earliest to latest.
df = df.set_index("date").sort_index()

# Create a calendar containing every day of 2023.
# This date range is specific to the CSV you uploaded.
calendar = pd.date_range(
    start="2023-01-01",
    end="2023-12-31",
    name="date"
)

# Add rows for missing dates.
# Their PM2.5 and AQI values stay NaN, meaning missing—not zero.
# This ensures neighboring rows represent consecutive calendar days.
df = df.reindex(calendar)

# Mark missing PM2.5 measurements:
# True means the measurement is missing; False means it exists.
df["measurement_missing"] = df["pm25"].isna()

# Turn the date index back into a regular column.
df = df.reset_index()

# Print a summary to check the cleaning results.
# Expected: 365 calendar days, 337 measured days, 28 missing days.
print("Total calendar days:", len(df))
print("Days with measurements:", df["pm25"].notna().sum())
print("Days missing measurements:", df["pm25"].isna().sum())

# Show the first 10 rows of the cleaned dataset.
display(df.head(10))

# Save the cleaned dataset as a new CSV in Google Drive.
# The original CSV is not overwritten.
df.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/veritas AI project/airaware_clean_daily.csv",
    index=False
)

Total calendar days: 365
Days with measurements: 337
Days missing measurements: 28


,date,pm25,aqi,measurement_missing
0,2023-01-01,5.6,31.0,False
1,2023-01-02,9.9,52.0,False
2,2023-01-03,15.3,63.0,False
3,2023-01-04,12.3,57.0,False
4,2023-01-05,12.5,57.0,False
5,2023-01-06,NaN,NaN,True
6,2023-01-07,NaN,NaN,True
7,2023-01-08,NaN,NaN,True
8,2023-01-09,NaN,NaN,True
9,2023-01-10,13.5,59.0,False
